# Iceberg Basics — 05: Partitioning (Hidden Partitioning)

## What you will learn
Iceberg's hidden partitioning is its most distinctive feature.
Partition transformations are applied automatically — your SQL never
needs to reference partition columns explicitly.

In this notebook:
1. What hidden partitioning is and why it matters
2. Partition transforms — years, months, days, hours, bucket, truncate
3. Partition pruning — how Spark skips irrelevant files
4. Multi-column partitioning
5. Reading partitioned tables — no partition filters needed in SQL
6. Comparing with Hive-style partitioning


In [ ]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

## Step 1 — Hidden Partitioning Transforms

Iceberg applies transformations to derive partition values from column values.
The original column is stored as-is; the partition is a derived value.

| Transform | Input | Partition value | Example |
|---|---|---|---|
| `years(ts)` | 2024-03-15 | 2024 | Annual partitions |
| `months(ts)` | 2024-03-15 | 2024-03 | Monthly partitions |
| `days(ts)` | 2024-03-15 | 2024-03-15 | Daily partitions |
| `hours(ts)` | 2024-03-15 14:30 | 2024-03-15-14 | Hourly partitions |
| `bucket(N, col)` | user_id=12345 | 12345 % N | Hash partitions |
| `truncate(W, col)` | "Electronics" | "Electro" | Prefix partitions |
| Identity | "US" | "US" | Exact value |


In [ ]:
# Monthly partitioning on order_date
spark.sql("DROP TABLE IF EXISTS local.icedb.part_orders")
spark.sql("""
    CREATE TABLE local.icedb.part_orders
    USING iceberg
    PARTITIONED BY (months(order_date), category)
    AS SELECT * FROM (VALUES(1)) t(x) WHERE false
""")
df.writeTo("local.icedb.part_orders").append()

print("Partitions (months × category):")
spark.sql("""
    SELECT partition, record_count, file_count,
           total_data_file_size_in_bytes/1024 AS size_kb
    FROM local.icedb.part_orders.partitions
    ORDER BY record_count DESC
    LIMIT 10
""").show(truncate=False)

## Step 2 — Partition Pruning: No Partition Column in WHERE


In [ ]:
# KEY DIFFERENCE FROM HIVE:
# Iceberg AUTOMATICALLY prunes partitions based on the transform
# You write SQL on the ORIGINAL column, not the partition column

runs = 3

# Full scan
t_full = []
for _ in range(runs):
    t0=time.time()
    spark.table("local.icedb.part_orders").agg(F.sum("revenue")).collect()
    t_full.append(time.time()-t0)

# Filtered — Iceberg derives partition filter from order_date predicate
t_pruned = []
for _ in range(runs):
    t0=time.time()
    spark.table("local.icedb.part_orders") \
         .filter(F.col("order_date").between("2024-06-01","2024-06-30")) \
         .agg(F.sum("revenue")).collect()
    t_pruned.append(time.time()-t0)

print("Hidden partitioning — filter on original column:")
print(f"  Full scan (all months)     : {sorted(t_full)[1]:.3f}s")
print(f"  WHERE order_date = June    : {sorted(t_pruned)[1]:.3f}s  ← Iceberg maps to months partition")
print(f"  Speedup                    : {sorted(t_full)[1]/sorted(t_pruned)[1]:.1f}x")
print()
print("In Hive-style partitioning you would write:")
print("  WHERE dt_year=2024 AND dt_month=6  ← ugly partition columns in SQL")
print("In Iceberg you write:")
print("  WHERE order_date BETWEEN '2024-06-01' AND '2024-06-30'  ← natural SQL")

## Step 3 — Bucket Partitioning for High-Cardinality Columns


In [ ]:
# Bucket partitioning for customer_id (50K distinct values)
# bucket(N, col) → assigns each value to one of N buckets via hash
spark.sql("DROP TABLE IF EXISTS local.icedb.bucket_orders")
spark.sql("""
    CREATE TABLE local.icedb.bucket_orders
    USING iceberg
    PARTITIONED BY (bucket(8, customer_id), months(order_date))
    AS SELECT * FROM (VALUES(1)) t(x) WHERE false
""")
df.writeTo("local.icedb.bucket_orders").append()

print("Bucket partitions (8 buckets × months):")
spark.sql("""
    SELECT partition, record_count
    FROM local.icedb.bucket_orders.partitions
    ORDER BY partition
    LIMIT 10
""").show(truncate=False)

# Bucket pruning: WHERE customer_id = X goes to exactly 1 bucket
t0 = time.time()
r = spark.table("local.icedb.bucket_orders") \
         .filter(F.col("customer_id") == 42) \
         .collect()
t_bucket = time.time() - t0

t0 = time.time()
r2 = spark.table("local.icedb.part_orders") \
          .filter(F.col("customer_id") == 42) \
          .collect()
t_no_bucket = time.time() - t0

print(f"\nWHERE customer_id = 42:")
print(f"  With bucket partition : {t_bucket:.3f}s  (reads 1/8 of files)")
print(f"  Without bucket        : {t_no_bucket:.3f}s  (reads all files)")

## Summary

```sql
-- Monthly partitioning (common for time-series)
PARTITIONED BY (months(event_ts))

-- Daily with categorical
PARTITIONED BY (days(event_ts), region)

-- Bucket for high-cardinality join column
PARTITIONED BY (bucket(16, user_id))

-- Truncate for string prefix
PARTITIONED BY (truncate(4, category))

-- Multiple transforms
PARTITIONED BY (years(order_date), bucket(8, customer_id))
```

### Hidden partitioning benefits
- No partition columns in SQL — just filter on original columns
- Iceberg derives the partition filter automatically
- Partition scheme can be changed without rewriting old data (partition evolution)
- No "wrong partition" bugs — cannot accidentally scan wrong partition
